# CC calculation (batch)

Put `cc_calc_core.py` in the **same folder** as this notebook. Edit thresholds there, then re-run cell 0 (`importlib.reload`).

Cells: **0** import · **1** print CONFIG · **2** choose **blocks** + folders · **3** run batch + Excel + plots.

In cell 2 a window asks which blocks to run (CC, cell properties, tau/Cm, spikelet). **Spikelet only** skips the other plots. AP start for spikelets is still the inflection method; the full Cell properties block is not required.

In [ ]:
# pip install pyabf pandas matplotlib scipy openpyxl
# openpyxl required for .xlsx export

import os
import importlib
from tkinter import Tk, filedialog

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyabf
from scipy.signal import find_peaks

import cc_calc_core as cc
importlib.reload(cc)
from cc_calc_core import (
    analyze_abf_file,
    build_file_summary_row,
    save_folder_summary_plot,
    save_folder_spikelet_over_time_plot,
    save_folder_spikelet_vs_vm_plot,
    save_folder_cc_norm_vs_vm_plot,
    save_folder_cc_vm_slope_over_time_plot,
    attach_cc_vm_slopes,
    format_excel_header_wrap,
    ask_analysis_blocks,
    resolve_analysis_blocks,
    spikelet_plots_dir,
    time_period_borders,
    cc_sweep_indices_direction,
    n_spikes_in_window,
    SAVE_QC_PLOTS,
    CELL_PROPS_PLOTS_SUBDIR,
    QC_PLOTS_SUBDIR,
    cell_props_plots_dir,
    qc_plots_dir,
    SAVE_CC_PLOTS,
    SAVE_CC_TRACE_PLOTS,
    SAVE_CC_VPOST_PLOTS,
    PLOT_DPI,
    CC_PLOTS_SUBDIR,
    CC_VM_FIT_LINEAR,
    SAVE_SPIKELET_PLOTS,
    SPIKELET_PLOTS_SUBDIR,
    SPIKELET_BASELINE_MS,
    SPIKELET_PEAK_MS,
    SPIKELET_NOISE_K,
    cc_plots_dir,
    compute_tau_cm,
    rin_for_channel,
    _save_tau_cm_qc_plot,
    _abf_stem,
)
from cc_calc_core import CC_SPIKE_HEIGHT, CC_SPIKE_DISTANCE, CC_MIN_DELTA_I_PA, CC_MIN_DELTA_V_MV
from cc_calc_core import (
    RIN_VMIN,
    RIN_VMAX,
    RIN_VMIN_FALLBACK,
    RIN_VMAX_FALLBACK,
    RIN_MIN_POINTS,
    CP_RIN_MIN_POINTS,
    CP_VREST_STAGES,
    EXPECTED_CHANNELS,
)

In [ ]:
# Thresholds: edit cc_calc_core.py, then re-run cell 0
print("CONFIG (cc_calc_core.py):")
print(f"  EXPECTED_CHANNELS = {EXPECTED_CHANNELS}")
print(f"  CC_SPIKE_HEIGHT = {CC_SPIKE_HEIGHT} mV")
print(f"  CC_SPIKE_DISTANCE = {CC_SPIKE_DISTANCE} samples")
print(f"  CC_MIN_DELTA_I_PA = {CC_MIN_DELTA_I_PA} pA")
print(f"  CC_MIN_DELTA_V_MV = {CC_MIN_DELTA_V_MV} mV")
print(f"  Rin Vm primary = ({RIN_VMIN}, {RIN_VMAX}) mV; fallback = ({RIN_VMIN_FALLBACK}, {RIN_VMAX_FALLBACK}) mV")
print(f"  V_rest I–V stages (need >={CP_RIN_MIN_POINTS} points; no V_rest from Rin_rel):")
for n_stage, vmin, vmax, tag in CP_VREST_STAGES:
    print(f"    {n_stage}) {tag}: {vmin}…{vmax} mV")
print(f"  Rin: abs linear then rel ΔV/ΔI; min points = {RIN_MIN_POINTS}")
print("  All_data: per-sweep CC only | File_summary: all file-level metrics")
print(f"  Cell-props plots (AP/Rin/tau): SAVE={SAVE_QC_PLOTS}, folder={CELL_PROPS_PLOTS_SUBDIR}")
print(f"  CC plots: SAVE={SAVE_CC_PLOTS}, traces={SAVE_CC_TRACE_PLOTS}, vpost={SAVE_CC_VPOST_PLOTS}, folder={CC_PLOTS_SUBDIR}")
print(f"  CC_norm vs Vm: linear fit, color = first→last ABF number; black = mean of all files")
print("  CC vs Vm slope over time: per-file linear slope (CC_norm / mV), both directions")
print("  Analysis blocks: choose in cell 2 (or ANALYSIS_BLOCKS). Spikelet uses AP start from inflections, not the Cell properties block.")
print("  Spikelet vs time: primary sweep only (first >=4 APs, else 3, else 2) — temporary")
print("  Spikelet vs Vm: sweep-mean ratio/delays vs Vm at spikelet begin (every sweep with >=2 APs)")
print(f"  PLOT_DPI = {PLOT_DPI} (lower = faster PNG writes)")
print(f"  Spikelets: SAVE={SAVE_SPIKELET_PLOTS}, baseline={SPIKELET_BASELINE_MS} ms, peak={SPIKELET_PEAK_MS} ms, k={SPIKELET_NOISE_K}*noise")
print("  Gj requires 0 < CC < 1; Excel default name = {folder}_CC_data.xlsx")

In [ ]:
# Cell 2 — choose analysis blocks, ABF folder, Excel path, and plot folders.
# Set variables below to skip the corresponding dialog.

ANALYSIS_BLOCKS = None             # e.g. {"cc": False, "cell_props": False, "tau_cm": False, "spikelets": True}
FOLDER_PATH = None                 # .abf folder
OUTPUT_EXCEL = None                # full .xlsx path
CELL_PROPS_PLOTS_PARENT = None     # parent for Cell_properties_plots/
CC_PLOTS_PARENT = None             # parent for CC_plots/


def _ask_directory(title, initialdir=None):
    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    root.lift()
    root.update()
    path = filedialog.askdirectory(title=title, initialdir=initialdir or "")
    root.destroy()
    return path or None


def _ask_save_excel(initialdir, initialfile):
    root = Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    root.lift()
    root.update()
    path = filedialog.asksaveasfilename(
        title="Save Excel results as…",
        defaultextension=".xlsx",
        filetypes=[("Excel workbook", "*.xlsx")],
        initialdir=initialdir or "",
        initialfile=initialfile,
    )
    root.destroy()
    return path or None


folder_path = FOLDER_PATH
output_excel = OUTPUT_EXCEL
cell_props_plots_parent = CELL_PROPS_PLOTS_PARENT
cc_plots_parent = CC_PLOTS_PARENT

analysis_blocks = resolve_analysis_blocks(ANALYSIS_BLOCKS)
if ANALYSIS_BLOCKS is None:
    try:
        analysis_blocks = ask_analysis_blocks()
    except Exception as exc:
        print("Analysis-block dialog skipped → all blocks on:", exc)
        analysis_blocks = resolve_analysis_blocks(None)
print("Analysis blocks:", {k: v for k, v in analysis_blocks.items()})
if analysis_blocks.get("spikelets") and not analysis_blocks.get("cell_props"):
    print("  Spikelet AP start = inflection (same method as cell properties); full cell-props block is off.")

if not folder_path:
    folder_path = _ask_directory("1/3 — Select folder with .abf files")

folder_name = os.path.basename(os.path.normpath(folder_path)) if folder_path else "CC"
default_excel_name = f"{folder_name}_CC_data.xlsx"

if folder_path and not output_excel:
    output_excel = _ask_save_excel(folder_path, default_excel_name)
    if not output_excel:
        output_excel = os.path.join(folder_path, default_excel_name)
        print("Excel dialog cancelled →", output_excel)

if folder_path and SAVE_QC_PLOTS and (
    analysis_blocks.get("cell_props") or analysis_blocks.get("tau_cm") or analysis_blocks.get("cc")
) and not cell_props_plots_parent:
    cell_props_plots_parent = _ask_directory(
        f"2/3 — Parent folder for '{CELL_PROPS_PLOTS_SUBDIR}' (AP / Rin / tau plots)",
        initialdir=folder_path,
    )
    if not cell_props_plots_parent:
        cell_props_plots_parent = folder_path
        print("Cell-props plots dialog cancelled → using ABF folder")

if folder_path and SAVE_CC_PLOTS and analysis_blocks.get("cc") and not cc_plots_parent:
    cc_plots_parent = _ask_directory(
        f"3/3 — Parent folder for '{CC_PLOTS_SUBDIR}' (CC QC plots)",
        initialdir=folder_path,
    )
    if not cc_plots_parent:
        cc_plots_parent = folder_path
        print("CC plots dialog cancelled → using ABF folder")

if folder_path:
    cell_props_plots_path = (
        cell_props_plots_dir(cell_props_plots_parent)
        if SAVE_QC_PLOTS and (
            analysis_blocks.get("cell_props") or analysis_blocks.get("tau_cm") or analysis_blocks.get("cc")
        ) else None
    )
    cc_plots_path = (
        cc_plots_dir(cc_plots_parent)
        if SAVE_CC_PLOTS and analysis_blocks.get("cc") else None
    )
    spikelet_plots_path = (
        spikelet_plots_dir(folder_path)
        if SAVE_SPIKELET_PLOTS and analysis_blocks.get("spikelets") else None
    )
    print("ABF folder:", folder_path)
    print("Excel:", output_excel)
    if cell_props_plots_path:
        print("Cell-props plots →", cell_props_plots_path)
    if cc_plots_path:
        print("CC plots →", cc_plots_path)
    if spikelet_plots_path:
        print("Spikelet plots →", spikelet_plots_path)
else:
    cell_props_plots_path = None
    cc_plots_path = None
    spikelet_plots_path = None
    print("No folder selected.")

In [ ]:
all_rows = []
summary_rows = []
spikelet_all_rows = []
spikelet_sweep_all_rows = []
n_plots_saved = 0
n_tau_plots = 0
n_cc_plots = 0

try:
    analysis_blocks
except NameError:
    analysis_blocks = resolve_analysis_blocks(None)

if folder_path:
    abf_files = sorted(f for f in os.listdir(folder_path) if f.lower().endswith(".abf"))
    if not abf_files:
        print("No .abf files in:", folder_path)
    else:
        for filename in abf_files:
            filepath = os.path.join(folder_path, filename)
            print(filename)
            try:
                plots_dir = cell_props_plots_path if SAVE_QC_PLOTS else None
                cc_dir = cc_plots_path if SAVE_CC_PLOTS else None
                sp_dir = spikelet_plots_path if SAVE_SPIKELET_PLOTS else None
                result = analyze_abf_file(
                    filepath, plots_dir=plots_dir, cc_plots_dir_path=cc_dir,
                    spikelet_plots_dir_path=sp_dir,
                    blocks=analysis_blocks,
                )
                detail, summary, plot_paths = result[0], result[1], result[2]
                spikelet_rows = result[3] if len(result) > 3 else []
                spikelet_sweep_rows = result[4] if len(result) > 4 else []
                all_rows.extend(detail)
                summary_rows.append(summary)
                spikelet_all_rows.extend(spikelet_rows)
                spikelet_sweep_all_rows.extend(spikelet_sweep_rows)
                n_plots_saved += len(plot_paths)
                n_tau_plots += sum(1 for p in plot_paths if p and "_tau_Cm" in str(p))
                n_cc_plots += sum(1 for p in plot_paths if p and "_CC_" in os.path.basename(str(p)))
                for ch in ("ch0", "ch2"):
                    tr = summary.get(f"tau_skip_reason_{ch}")
                    cs = summary.get(f"Cm_skip_reason_{ch}")
                    vr = summary.get(f"V_rest_skip_reason_{ch}")
                    vn = summary.get(f"V_rest_note_{ch}")
                    if tr or cs:
                        print(f"  {ch}: tau={summary.get(f'tau_ms_{ch}')} ms, Cm={summary.get(f'Cm_pF_{ch}')} pF")
                        if tr:
                            print(f"    tau_skip: {tr}")
                        if cs:
                            print(f"    Cm_skip: {cs}")
                    vs = summary.get(f"V_rest_stage_{ch}")
                    if vr:
                        print(f"  {ch} V_rest skip: {vr}")
                    elif vs and vs != "1_primary":
                        if vn:
                            print(f"  {ch} V_rest note: {vn}")
                        else:
                            print(f"  {ch} V_rest stage: {vs}")
            except Exception as exc:
                print(f"  ERROR: {exc}")
                err = str(exc)
                all_rows.append({
                    "file": filename,
                    "recording_datetime": None,
                    "CC_skip_reason": err,
                })
                summary_rows.append(
                    build_file_summary_row(
                        filename, None, None, None, None, None, err, err,
                        None, None, 0, 0, None, None, err, err, 0, 0,
                        file_skip_reason=err,
                    )
                )

        if summary_rows:
            if analysis_blocks.get("cc"):
                attach_cc_vm_slopes(summary_rows, all_rows)
            with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
                pd.DataFrame(all_rows).to_excel(writer, sheet_name="All_data", index=False)
                pd.DataFrame(summary_rows).to_excel(writer, sheet_name="File_summary", index=False)
                if spikelet_all_rows:
                    pd.DataFrame(spikelet_all_rows).to_excel(
                        writer, sheet_name="Spikelets", index=False
                    )
                if spikelet_sweep_all_rows:
                    pd.DataFrame(spikelet_sweep_all_rows).to_excel(
                        writer, sheet_name="Spikelet_sweeps", index=False
                    )
                try:
                    format_excel_header_wrap(writer.book)
                except Exception as exc:
                    print(f"  Excel header wrap skipped: {exc}")
            print("Saved Excel:", output_excel)
            print("  All_data rows:", len(all_rows))
            print("  File_summary rows:", len(summary_rows))
            if spikelet_all_rows:
                print("  Spikelets rows:", len(spikelet_all_rows))
            if spikelet_sweep_all_rows:
                print("  Spikelet_sweeps rows:", len(spikelet_sweep_all_rows))
            folder_name = os.path.basename(os.path.normpath(folder_path))
            if analysis_blocks.get("cc") or analysis_blocks.get("cell_props"):
                folder_plot = os.path.join(
                    folder_path,
                    f"{folder_name}_CC_Gj_Rin_Vm_over_time.png",
                )
                try:
                    saved = save_folder_summary_plot(
                        summary_rows,
                        folder_plot,
                        title=folder_name,
                    )
                    if saved:
                        print("  Folder CC/Gj/Rin/Vm vs time:", saved)
                    else:
                        print("  Folder CC/Gj/Rin/Vm vs time: skipped (no recording datetime / values)")
                except Exception as exc:
                    print(f"  Folder CC/Gj/Rin/Vm vs time error: {exc}")
            if analysis_blocks.get("spikelets"):
                spikelet_time_plot = os.path.join(
                    folder_path,
                    f"{folder_name}_spikelet_over_time.png",
                )
                try:
                    saved_sp = save_folder_spikelet_over_time_plot(
                        summary_rows,
                        spikelet_time_plot,
                        title=folder_name,
                        spikelet_rows=spikelet_all_rows,
                        spikelet_sweep_rows=spikelet_sweep_all_rows,
                    )
                    if saved_sp:
                        print("  Folder spikelet vs time:", saved_sp)
                        print("  look in ABF folder:", saved_sp)
                        if spikelet_plots_path:
                            import shutil
                            dest = os.path.join(spikelet_plots_path, os.path.basename(saved_sp))
                            shutil.copy2(saved_sp, dest)
                            print("  also copied to Spikelet_plots:", dest)
                    else:
                        print("  Folder spikelet vs time: skipped (no File_summary rows)")
                except Exception as exc:
                    print(f"  Folder spikelet vs time error: {exc}")
                    import traceback
                    traceback.print_exc()
            if analysis_blocks.get("cc"):
                cc_norm_plot = os.path.join(
                    folder_path,
                    f"{folder_name}_CC_norm_vs_Vm.png",
                )
                try:
                    saved_cn = save_folder_cc_norm_vs_vm_plot(
                        all_rows,
                        cc_norm_plot,
                        title=folder_name,
                        summary_rows=summary_rows,
                    )
                    if saved_cn:
                        print("  Folder CC_norm vs Vm:", saved_cn)
                    else:
                        print("  Folder CC_norm vs Vm: skipped (no valid CC_norm points)")
                except Exception as exc:
                    print(f"  Folder CC_norm vs Vm error: {exc}")
                    import traceback
                    traceback.print_exc()
                slope_time_plot = os.path.join(
                    folder_path,
                    f"{folder_name}_CC_vs_Vm_slope_over_time.png",
                )
                try:
                    saved_sl = save_folder_cc_vm_slope_over_time_plot(
                        all_rows,
                        slope_time_plot,
                        title=folder_name,
                        summary_rows=summary_rows,
                    )
                    if saved_sl:
                        print("  Folder CC vs Vm slope over time:", saved_sl)
                    else:
                        print("  Folder CC vs Vm slope over time: skipped (need >=3 CC_norm points/file)")
                except Exception as exc:
                    print(f"  Folder CC vs Vm slope over time error: {exc}")
                    import traceback
                    traceback.print_exc()
            if analysis_blocks.get("spikelets"):
                spikelet_vm_plot = os.path.join(
                    folder_path,
                    f"{folder_name}_spikelet_vs_Vm.png",
                )
                try:
                    saved_svm = save_folder_spikelet_vs_vm_plot(
                        spikelet_sweep_all_rows,
                        spikelet_vm_plot,
                        title=folder_name,
                        summary_rows=summary_rows,
                        spikelet_rows=spikelet_all_rows,
                    )
                    if saved_svm:
                        print("  Folder spikelet vs Vm:", saved_svm)
                        print("  look in ABF folder:", saved_svm)
                        if spikelet_plots_path:
                            import shutil
                            dest = os.path.join(spikelet_plots_path, os.path.basename(saved_svm))
                            shutil.copy2(saved_svm, dest)
                            print("  also copied to Spikelet_plots:", dest)
                    else:
                        print("  Folder spikelet vs Vm: skipped (no sweep-mean + Vm_begin points)")
                except Exception as exc:
                    print(f"  Folder spikelet vs Vm error: {exc}")
                    import traceback
                    traceback.print_exc()
            if SAVE_QC_PLOTS and cell_props_plots_path:
                print("  Cell-props plots:", n_plots_saved, "files →", cell_props_plots_path)
                print("  (look for *_AP.png, *_IV_Rin.png, *_tau_Cm.png)")
            if SAVE_CC_PLOTS and cc_plots_path:
                print("  CC plots:", n_cc_plots, "files →", cc_plots_path)
                print("  (look for *_CC_traces.png, *_CC_vs_Vpost.png)")
            if SAVE_SPIKELET_PLOTS and spikelet_plots_path:
                print("  Spikelet plots →", spikelet_plots_path)
                print("  (look for *_ch0-ch2_spikelets.png, *_ch2-ch0_spikelets.png)")
        else:
            print("No results.")
else:
    print("No folder selected.")